In [1]:
# Ambiente de motores
#Librerias requeridas
import random

class MDP:
    def __init__(self, table, dimensions, initial_state):
        self.nrows, self.ncols = dimensions
        self.transitions = [[0 for _ in range(self.ncols)] for _ in range(self.nrows)]
        self.rewards = [[0 for _ in range(self.ncols)] for _ in range(self.nrows)]

        for i in range(self.nrows):
            for j in range(self.ncols):
                state = table[i][j]
                if state is not None:
                    list_transitions = []
                    list_rewards = []
                    for inf in state:
                        p, s, r = inf
                        list_transitions.append((p,s))
                        list_rewards.append((r,s))
                    self.transitions[i][j] = list_transitions
                    self.rewards[i][j] = list_rewards
                else:
                    self.transitions[i][j] = None
                    self.rewards[i][j] = None
        self.initial_state = initial_state
        self.state = self.initial_state

    def get_current_state(self):
        return self.state

    def get_possible_actions(self, state:int) -> list[str]:
        actions = ['slow', 'fast']
        if state == 2:
            actions = []
        return actions

    def get_action_index(self, action: str) -> int:
        actions = ['slow', 'fast']
        index = 0
        for a in actions:
            if action == a:
                return index
            index += 1

    def get_possible_states(self, state, action) -> tuple[list[float], list[int], list[int]]:
        action_index = self.get_action_index(action)
        new_state = None
        rewards=[]
        states=[]
        probabilities = []
        for i in range(len(self.transitions[state][action_index])):
            prob, new_state = self.transitions[state][action_index][i]
            reward, _ = self.rewards[state][action_index][i]
            probabilities.append(prob)
            rewards.append(reward)
            states.append(new_state)
        return probabilities, rewards, states

    def simulate_action(self, state, action):
        act = -1
        if action == 'slow':
            act = 0
        else:
            act = 1
        start_state_action = self.transitions[state][act]
        prob = 0
        index = -1
        new_state = -1
        for op in start_state_action:
            r = random.random()
            if prob < r and r <= prob + op[0]:
                new_state = op[1]
            prob += op[0]
            index += 1
        return self.rewards[state][act][index][0], new_state

    def do_action(self, action):
        act = -1
        state = self.state
        if action == 'slow':
            act = 0
        else:
            act = 1
        start_state_action = self.transitions[state][act]
        prob = 0
        index = -1
        for op in start_state_action:
            r = random.random()
            if prob < r and r <= prob + op[0]:
                self.state = op[1]
            prob += op[0]
            index += 1
        return self.rewards[state][act][index][0], self.state

    def reset(self):
        self.state = self.initial_state

    def is_terminal(self, state):
        return self.get_possible_actions(state) == []

In [2]:
# Definición de las librerías a utilizar
import random

class PolicyIteration():
    def __init__(self, mdp: MDP, discount: float =0.9, iterations:int=64):
        if not isinstance(mdp, MDP):
            raise TypeError("mdp debe ser una instancia de la clase MDP.")
        if not isinstance(discount, float):
            raise TypeError("discount debe ser un número decimal (float).")
        if not (0 < discount <= 1):
            raise ValueError("El factor de descuento debe estar entre 0 (excluido) y 1 (incluido).")
        if not isinstance(iterations, int):
            raise TypeError("iterations debe ser un entero.")

        self.mdp = mdp
        self.discount = discount
        self.iterations = iterations

        self.values = {}
        self.policy = {}

        for state in range(len(self.mdp.transitions)):
            if not self.mdp.is_terminal(state):
                actions = self.mdp.get_possible_actions(state)
                if actions:
                    self.policy[state] = random.choice(actions)
                else:
                    self.policy[state] = None
            else:
                self.policy[state] = None

    def set_policy(self, policy:list[str]):
        #Función de actualización de la política usada para las pruebas
        self.policy = policy

    def get_policy(self, state:int) -> str:
        if self.mdp.is_terminal(state):
            return None
        if state in range(len(self.policy)) and self.policy[state] is not None:
            return self.policy[state]
        possible_actions = self.mdp.get_possible_actions(state)
        if possible_actions:
            return possible_actions[0]
        return None

    def get_value(self, state:int) -> float:
        valid_states = list(range(len(self.mdp.transitions)))
        if state not in valid_states:
            raise ValueError(f"Estado inválido: {state}. Los estados válidos son {valid_states}.")
        return self.values.get(state, 0)

    def get_action(self, state:int) -> str:
        if self.mdp.is_terminal(state):
            return None
        action = self.policy.get(state, None)
        valid_actions = self.mdp.get_possible_actions(state)
        if action not in valid_actions:
            return None
        return action

    def compute_new_action(self, state:int) -> str:
        if self.mdp.is_terminal(state):
            self.policy[state] = None
            return None

        best_value = float('-inf')
        best_action = None
        possible_actions = self.mdp.get_possible_actions(state)

        for action in possible_actions:
            value = self.value_evaluation(state, action)
            if value > best_value:
                best_value = value
                best_action = action

        self.policy[state] = best_action
        return best_action

    def value_evaluation(self, state:int, action:str) -> float:
        valid_states = list(range(len(self.mdp.transitions)))
        if state not in valid_states:
            raise ValueError(f"Estado inválido: {state}. Los estados válidos son {valid_states}.")
        if self.mdp.is_terminal(state):
            return 0.0
        probs, rewards, next_states = self.mdp.get_possible_states(state, action)
        value = 0.0
        for p, r, s_prime in zip(probs, rewards, next_states):
            value += p * (r + self.discount * self.get_value(s_prime))
        return value

    def compute_new_value(self, state:int) -> tuple[float,str]:
        valid_states = list(range(len(self.mdp.transitions)))
        if state not in valid_states:
            raise ValueError(f"Estado inválido: {state}. Los estados válidos son {valid_states}.")
        if self.mdp.is_terminal(state):
            return 0.0, None
        best_value = float('-inf')
        best_action = None
        possible_actions = self.mdp.get_possible_actions(state)
        for action in possible_actions:
            value = self.value_evaluation(state, action)
            if value > best_value:
                best_value = value
                best_action = action
        if best_action is None:
            return 0.0, None
        return best_value, best_action


    def policy_evaluation(self, new_policy:dict[int, str]) -> bool:
        if not isinstance(new_policy, dict):
            raise TypeError("La nueva política debe ser un diccionario con formato {estado: acción}.")
        try:
            states = [i for i in range(len(self.mdp.transitions)) if self.mdp.transitions[i] is not None]
        except AttributeError:
            raise AttributeError("El MDP no tiene información de estados accesible.")

        for state in states:
            current_action = self.policy.get(state, None)
            new_action = new_policy.get(state, None)
            valid_actions = self.mdp.get_possible_actions(state)
            if new_action not in valid_actions and new_action is not None:
                raise ValueError(f"Acción '{new_action}' no válida para el estado {state}. "
                                 f"Acciones posibles: {valid_actions}")
            if current_action != new_action:
                return True
        return False

    def policy_iteration(self) -> tuple[bool, dict[int, str]]:
        all_states = [state for state in range(self.mdp.nrows) if not self.mdp.is_terminal(state)]
        old_policy = self.policy.copy()
        EVAL_ITERATIONS = 10
        for _ in range(EVAL_ITERATIONS):
            new_values = self.values.copy()
            for state in all_states:
                action = self.policy.get(state)
                new_v_s = self.value_evaluation(state, action)
                new_values[state] = new_v_s
            self.values = new_values

        for state in all_states:
            self.compute_new_action(state)

        new_policy = self.policy.copy()
        policy_changed = False
        for state in all_states:
            if old_policy.get(state) != self.policy.get(state):
                policy_changed = True
                break
        convergencia = not policy_changed
        return convergencia, new_policy

    def run_policy_iteration(self) -> tuple[dict[float], dict[str]]:
        done = False
        i = 1
        policy = {}
        while not done and i <= self.iterations:
            done, policy = self.policy_iteration()
            i += 1
        if done:
            print(f"La política converge en {i} iteraciones")
            print(f"Política : {policy}")
        return self.values, self.policy


In [3]:
#Ambiente general de prueba para todos los casos siguientes

board = [[[(1,0,1)], [(0.5, 0, 2), (0.5,1,2)]],
         [[(0.5, 0, 1), (0.5,1,1)], [(1, 2, -10)]],
         [None, None]]

env = MDP(board, (3,2), 0)

In [11]:
#Pruebas de ejecución del agente

a = PolicyIteration(env, 0.9, 100)

a.run_policy_iteration()

La política converge en 2 iteraciones
Política : {0: 'fast', 1: 'slow', 2: None}


({0: 10.269823398500002, 1: 9.269823398500002},
 {0: 'fast', 1: 'slow', 2: None})